In [63]:
import os
import pandas as pd
import numpy as np

In [64]:
#load the dataset
df = pd.read_csv('./data/imdb_master.csv', encoding='latin-1')
df = df[['review', 'label']]

In [65]:
# look at dataset
print(df.head())

                                              review label
0  Once again Mr. Costner has dragged out a movie...   neg
1  This is an example of why the majority of acti...   neg
2  First of all I hate those moronic rappers, who...   neg
3  Not even the Beatles could write songs everyon...   neg
4  Brass pictures (movies is not a fitting word f...   neg


In [66]:
df = df[df["label"].isin(["neg", "pos"])].copy()

df["label_num"] = df["label"].map({
    "neg": 0,
    "pos": 1
})

In [67]:
df.describe()

,label_num
count,50000.000000
mean,0.500000
std,0.500005
min,0.000000
25%,0.000000
50%,0.500000
75%,1.000000
max,1.000000


In [68]:
#drop na values
df = df.dropna()

In [69]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   label      50000 non-null  str  
 2   label_num  50000 non-null  int64
dtypes: int64(1), str(2)
memory usage: 1.1 MB


In [70]:
#create a new column 'label_num' to convert the labels to numerical values
df['label_num'] = df['label'].map({'neg': 0, 'pos': 1})

In [71]:
df['label_num'].value_counts()

label_num
0    25000
1    25000
Name: count, dtype: int64

In [72]:
df.head()

,review,label,label_num
0,Once again Mr. Costner has dragged out a movie...,neg,0
1,This is an example of why the majority of acti...,neg,0
2,"First of all I hate those moronic rappers, who...",neg,0
3,Not even the Beatles could write songs everyon...,neg,0
4,Brass pictures (movies is not a fitting word f...,neg,0


In [73]:
print(df.shape)
print(df.head())
print(df.columns)
print(df.info())
print(df['label'].value_counts())

(50000, 3)
                                              review label  label_num
0  Once again Mr. Costner has dragged out a movie...   neg          0
1  This is an example of why the majority of acti...   neg          0
2  First of all I hate those moronic rappers, who...   neg          0
3  Not even the Beatles could write songs everyon...   neg          0
4  Brass pictures (movies is not a fitting word f...   neg          0
Index(['review', 'label', 'label_num'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   label      50000 non-null  str  
 2   label_num  50000 non-null  int64
dtypes: int64(1), str(2)
memory usage: 1.1 MB
None
label
neg    25000
pos    25000
Name: count, dtype: int64


In [74]:
df = df.sample(1000, random_state=42).reset_index(drop=True)  # Shuffle the dataset

In [75]:
print(df.shape)

(1000, 3)


In [78]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_num"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label_num"]
)

In [80]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (800, 3)
Validation: (100, 3)
Test: (100, 3)


In [84]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text.split()

In [85]:
text = "This is a sample text with HTML tags <b>bold</b> and special characters! @#"

print("Original text:", text)
tokenized_text = tokenize(text)
print("Tokenized text:", tokenized_text)

Original text: This is a sample text with HTML tags <b>bold</b> and special characters! @#
Tokenized text: ['this', 'is', 'a', 'sample', 'text', 'with', 'html', 'tags', 'bboldb', 'and', 'special', 'characters']


In [86]:
from collections import Counter

counter = Counter()

for review in train_df["review"]:
    tokens = tokenize(review)
    counter.update(tokens)

In [88]:
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, frequency in counter.items():
    vocab[word] = len(vocab)

In [89]:
print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:20])

Vocabulary size: 18026
[('<PAD>', 0), ('<UNK>', 1), ('i', 2), ('could', 3), ('see', 4), ('this', 5), ('film', 6), ('is', 7), ('super', 8), ('he', 9), ('didnt', 10), ('surprise', 11), ('to', 12), ('oneself', 13), ('when', 14), ('so', 15), ('that', 16), ('it', 17), ('was', 18), ('taking', 19)]


In [90]:
def encode(text, vocab):
    tokens = tokenize(text)

    return [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

In [91]:
review = "This product is amazing"

encoded = encode(review, vocab)

print(encoded)

[5, 5091, 7, 1427]


In [93]:
MAX_LEN = 200

def pad_sequence(sequence, max_len, pad_id=0):

    if len(sequence) < max_len:
        sequence = sequence + [pad_id] * (max_len - len(sequence))

    else:
        sequence = sequence[:max_len]

    return sequence

In [94]:
sequence = [5, 5091, 7, 1427]

padded = pad_sequence(sequence, MAX_LEN)

print(len(padded))
print(padded[:10])

200
[5, 5091, 7, 1427, 0, 0, 0, 0, 0, 0]


In [95]:
import torch
from torch.utils.data import Dataset

In [96]:
class ReviewDataset(Dataset):

    def __init__(self, dataframe, vocab, max_len):
        self.reviews = dataframe["review"].values
        self.labels = dataframe["label_num"].values

        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, index):

        review = self.reviews[index]
        label = self.labels[index]

        # text → tokens → IDs
        encoded = encode(review, self.vocab)

        # make every sequence the same length
        encoded = pad_sequence(
            encoded,
            self.max_len
        )

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.float)
        )

In [97]:
MAX_LEN = 200

train_dataset = ReviewDataset(
    train_df,
    vocab,
    MAX_LEN
)

val_dataset = ReviewDataset(
    val_df,
    vocab,
    MAX_LEN
)

test_dataset = ReviewDataset(
    test_df,
    vocab,
    MAX_LEN
)

In [98]:
x, y = train_dataset[0]

print("Input:", x)
print("Input shape:", x.shape)
print("Label:", y)

Input: tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
        20, 21, 22, 23,  5, 24, 25, 26, 27, 28, 29, 12, 22, 30, 10, 31, 22, 20,
        32, 17,  7, 33, 34, 35, 36, 37, 38, 39, 40,  2, 41, 42,  5,  6, 14,  2,
        18, 22, 43,  2, 44, 45, 30, 15, 16, 17, 18, 46, 17, 47, 48, 16, 34, 49,
        29, 18, 19, 20, 50, 51, 12, 17, 21, 52, 49, 53, 54, 17,  7, 55, 56, 57,
        22, 58, 59, 60, 61, 62, 63, 64, 12, 65, 66, 67, 68, 69, 22, 70,  2, 40,
        71, 72, 21, 22, 73, 74, 57, 75, 67, 76,  2, 77, 49, 78, 79, 80, 67, 81,
        82, 83, 25, 84, 66, 22, 85, 86, 87, 51, 88, 89, 33, 34, 90, 91, 32, 60,
        92,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0])
Input shape: torch.Size([200])
Label: tensor(1.)


In [99]:
from torch.utils.data import DataLoader


BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [100]:
x, y = next(iter(train_loader))

print("Input shape:", x.shape)
print("Label shape:", y.shape)

Input shape: torch.Size([32, 200])
Label shape: torch.Size([32])


In [102]:

from torch import nn


class SentimentRNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            1
        )

    def forward(self, x):

        # [batch, sequence]
        embedded = self.embedding(x)

        # [batch, sequence, embedding_dim]
        output, hidden = self.rnn(embedded)

        # Take the final hidden state
        last_hidden = hidden[-1]

        # [batch, hidden_dim] → [batch, 1]
        output = self.fc(last_hidden)

        return output.squeeze(1)

In [103]:
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

model = SentimentRNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

print(model)

SentimentRNN(
  (embedding): Embedding(18026, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)
